In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpila circuitos de forma remota con el Qiskit Transpiler Service

> **Danger:** A partir del 18 de julio de 2025, el servicio está siendo migrado para dar soporte a la nueva IBM Quantum&reg; Platform y no está disponible. Para los pases de IA, puedes usar el [modo local](/guides/ai-transpiler-passes#run-the-ai-transpiler-passes-locally-or-on-the-cloud).
> 
>     El servicio es una versión beta, sujeta a cambios.
>     Si tienes comentarios o quieres contactar al equipo de desarrollo, usa este canal de [Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

El Qiskit Transpiler Service proporciona capacidades de transpilación en la nube. Además de las capacidades del transpilador local de Qiskit, tus tareas de transpilación pueden beneficiarse tanto de los recursos en la nube de IBM Quantum como de los pases de transpilador potenciados por IA.

El Qiskit Transpiler Service ofrece una biblioteca de Python para integrar este servicio y sus capacidades de forma transparente en tus patrones y flujos de trabajo actuales de Qiskit. Este servicio está disponible únicamente para usuarios del IBM Quantum Premium Plan, Flex Plan y On-Prem (a través de la API de IBM Quantum Platform).

<span id="install-transpiler-service"></span>
## Instala el paquete qiskit-ibm-transpiler
Para usar el Qiskit Transpiler Service, instala el paquete `qiskit-ibm-transpiler`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="false",
    optimization_level=3,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

El paquete se autentica automáticamente usando tus [credenciales de IBM Quantum Platform](/guides/cloud-setup), de forma alineada con cómo [Qiskit Runtime las gestiona](/guides/initialize-account):
- Variable de entorno: `QISKIT_IBM_TOKEN`
- Archivo de configuración `~/.qiskit/qiskit-ibm.json` (bajo la sección `default-ibm-quantum`).

*Nota*: Este paquete requiere Qiskit SDK v1.X.

## Opciones de transpilación de qiskit-ibm-transpiler
- `backend_name` (opcional, str) - El nombre de un backend tal como lo esperaría QiskitRuntimeService (por ejemplo, `ibm_torino`). Si se establece, el método de transpilación utiliza el layout del backend especificado para la operación de transpilación. Si se establece cualquier otra opción que afecte a esta configuración, como `coupling_map`, los ajustes de `backend_name` son reemplazados.
- `coupling_map` (opcional, List[List[int]]) - Una lista de coupling map válida (por ejemplo, [[0,1],[1,2]]). Si se establece, el método de transpilación utiliza este coupling map para la operación de transpilación. Si se define, reemplaza cualquier valor especificado para `target`.
- `optimization_level` (int) - El nivel de optimización potencial que se aplicará durante el proceso de transpilación. Los valores válidos son [1,2,3], donde 1 es la menor optimización (y la más rápida) y 3 es la mayor optimización (y la que más tiempo requiere).
- `ai` ("true", "false", "auto") - Si se deben usar capacidades potenciadas por IA durante la transpilación. Las capacidades disponibles con IA pueden ser para pases de transpilación `AIRouting` u otros métodos de síntesis potenciados por IA. Si este valor es `"true"`, el servicio aplica diferentes pases de transpilación con IA según el `optimization_level` solicitado. Si es `"false"`, usa las últimas funciones de transpilación de Qiskit sin IA. Por último, si es `"auto"`, el servicio decide si aplicar los pases heurísticos estándar de Qiskit o los pases con IA en función de tu circuito.
- `qiskit_transpile_options` (dict) - Un objeto de diccionario de Python que puede incluir cualquier otra opción válida en el [método `transpile()` de Qiskit](defaults-and-configuration-options). Si `qiskit_transpile_options` incluye `optimization_level`, se descarta en favor del `optimization_level` especificado como parámetro de entrada. Si `qiskit_transpile_options` incluye alguna opción no reconocida por el método `transpile()` de Qiskit, la biblioteca lanza un error.

Para más información sobre los métodos disponibles de `qiskit-ibm-transpiler`, consulta la [referencia de la API de qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler). Para aprender más sobre la API del servicio, consulta la [documentación de la REST API del Qiskit Transpiler Service.](https://docs.quantum.ibm.com/api/qiskit-transpiler-service-rest)

## Ejemplos
Los siguientes ejemplos demuestran cómo transpilar circuitos usando el Qiskit Transpiler Service con diferentes parámetros.

1. Crea un circuito y llama al Qiskit Transpiler Service para transpilarlo con `ibm_torino` como `backend_name`, 3 como `optimization_level` y sin usar IA durante la transpilación.

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="true",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

*Nota*: solo puedes usar dispositivos de backend_name a los que tengas acceso con tu cuenta de IBM Quantum. Además de `backend_name`, el `TranspilerService` también admite `coupling_map` como parámetro.

2. Genera un circuito similar y transpílalo, solicitando capacidades de transpilación con IA al establecer el indicador `ai` en `True`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="auto",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

3. Genera un circuito similar y transpílalo dejando que el servicio decida si usar los pases de transpilación potenciados por IA.